## Caso práctico: EDA de ventas de una cadena retail

La empresa **RetailNova Perú** comercializa productos tecnológicos y accesorios a través de cuatro tiendas:
- Lima
- Arequipa
- Trujillo
- Cusco

La gerencia ha entregado al área de análisis de datos un archivo con las ventas de los últimos seis meses. El problema es que la gerencia tiene los datos, pero no sabe exactamente:

- qué sucursal tiene mejor desempeño;
- qué productos generan mayores ingresos;
- si los descuentos están ayudando a vender más;
- qué métodos de pago utilizan los clientes;
- si existen datos incorrectos o anomalías;
- qué factores están asociados con las devoluciones;
- qué conclusiones comerciales pueden obtenerse.

### Objetivo general

La gerencia de RetailNova solicita realizar un análisis exploratorio de sus operaciones comerciales con el objetivo de identificar patrones de ventas, comportamiento de clientes, desempeño de sucursales, posibles problemas de calidad de datos y oportunidades comerciales.

In [ ]:
# 1. Cargar librerias necesarias para el EDA
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# 2. Cargar datos de ventas
df_ventas = pd.read_csv("./data/ventas_retail.csv", encoding="utf-8")
df_ventas.head()

In [ ]:
# 3. Obtener información general del dataset
df_ventas.info()

In [ ]:
# 4. Análisis de valores nulos
df_ventas.isna().sum().sort_values(ascending = False)

In [ ]:
# 5. Estadísticas descriptivas del dataset
df_ventas.describe()

Conclusiones de las estadísticas de datos númericos:
- Unidades vendidas mayores
- Descuentos exagerados
- Edad del cliente exagerada (no real)
- Demasiado tiempo de atención

In [ ]:
# 6. Revisión de anomalías detectadas
# Unidades vendidas mayor a 20
df_anomalia_cantidad = df_ventas[df_ventas["cantidad"] > 20]
print("Unidades vendidas mayor a 20: ", df_anomalia_cantidad["cantidad"].count())
df_anomalia_cantidad.head()

In [ ]:
# Descuentos mayores a 30%
df_anomalia_descuento = df_ventas[df_ventas["descuento_pct"] > 30]
print("Descuentos mayores a 30%: ", df_anomalia_descuento["descuento_pct"].count())
df_anomalia_descuento.head()

In [ ]:
# Clientes con edad mayor a 85
df_anomalia_edad = df_ventas[df_ventas["edad_cliente"] > 85]
print("Clientes con edad mayor a 85: ", df_anomalia_edad["edad_cliente"].count())
df_anomalia_edad.head()

In [ ]:
# Tiempo de atencion mayor a 30 minutos (conteo)
df_anomalia_tiempo = df_ventas[df_ventas["tiempo_atencion_min"] > 45]
print("Tiempo de atencion mayor a 45 minutos: ", df_anomalia_tiempo["tiempo_atencion_min"].count())
df_anomalia_tiempo.head()

In [ ]:
# 7. Analisis de duplicados
df_duplicados = df_ventas[df_ventas.duplicated()]
print("Registros duplicados: ", df_duplicados.shape[0])
df_duplicados.head()

In [ ]:
# Funcion para mostrar graficos con datos categoricos
def graficos_eda_categoricos(cat):
    #Calculamos el número de filas que necesitamos
    from math import ceil
    filas = ceil(cat.shape[1] / 2)

    #Definimos el gráfico
    f, ax = plt.subplots(nrows = filas, ncols = 2, figsize = (16, filas * 6))

    #Aplanamos para iterar por el gráfico como si fuera de 1 dimensión en lugar de 2
    ax = ax.flat

    #Creamos el bucle que va añadiendo gráficos
    for cada, variable in enumerate(cat):
        cat[variable].value_counts().plot.barh(ax = ax[cada])
        ax[cada].set_title(variable, fontsize = 12, fontweight = "bold")
        ax[cada].tick_params(labelsize = 12)

In [ ]:
# 8. Mostrar graficos de datos categoricos
graficos_eda_categoricos(df_ventas.select_dtypes(include=["object"]))

In [ ]:
# 9. Normalizar categorias (sucursal, categoria, metodo_pago y tipo_cliente) convirtiendo a mayusculas y eliminando espacios en blanco
df_ventas["sucursal"] = df_ventas["sucursal"].str.upper().str.strip()
df_ventas["categoria"] = df_ventas["categoria"].str.upper().str.strip()
df_ventas["metodo_pago"] = df_ventas["metodo_pago"].str.upper().str.strip()
df_ventas["tipo_cliente"] = df_ventas["tipo_cliente"].str.upper().str.strip()

In [ ]:
# Mostramos el gráfico con las columnas normalizadas
graficos_eda_categoricos(df_ventas[["sucursal", "categoria", "metodo_pago", "tipo_cliente"]])

Conclusiones de los gráficos obtenidos:

- Cusco es la sucursal con mayor número de ventas reportado
- Los clientes VIP son los que menos compran con frecuencia
- Los métodos de pago es variable
- Laptops y monitores con la categoría de producto más vendido

In [ ]:
# 10. Crear columnas adicionales para el análisis
# Crear importe de venta
df_ventas["importe_venta"] = df_ventas["cantidad"] * df_ventas["precio_unitario"]

# Crear columna de descuento
df_ventas["descuento"] = df_ventas["importe_venta"] * df_ventas["descuento_pct"] / 100

# Crear columna de importe neto
df_ventas["importe_neto"] = df_ventas["importe_venta"] - df_ventas["descuento"]

In [ ]:
# 11. Calcular indicadores de venta
# Calcular el importe neto por sucursal y mostrarlo en un gráfico de pie (con etiqueta de porcentaje y el monto en miles)
importe_neto_por_sucursal = df_ventas.groupby("sucursal")["importe_neto"].sum()
importe_neto_por_sucursal.plot.pie(autopct=lambda p: '{:.1f}%\n${:,.0f}k'.format(p, p * importe_neto_por_sucursal.sum() / 1000), figsize=(8, 8), title="Importe Neto por Sucursal")

In [ ]:
# Mostrar el top 5 de productos que generan mayores ingresos (mostrar monto en el grafico de barras)
top_5_productos = df_ventas.groupby("producto")["importe_neto"].sum().nlargest(5)
top_5_productos.plot.bar(figsize=(10, 6), title="Top 5 Productos por Ingresos", ylabel="Ingresos", rot=45, legend=False, fontsize=12)
for index, value in enumerate(top_5_productos):
    plt.text(index, value, f"${value:,.0f}", ha='center', va='bottom', fontsize=12)


In [ ]:
# Mostrar la cantidad de ventas con descuento y sin descuento por sucursal en una grafico de barras (con series)
df_ventas["tiene_descuento"] = df_ventas["descuento"] > 0

ventas_con_y_sin_descuento = df_ventas.groupby(["sucursal", "tiene_descuento"])["importe_neto"].count().unstack()
ventas_con_y_sin_descuento.plot.bar(figsize=(10, 6), title="Ventas con y sin Descuento por Sucursal", ylabel="Cantidad de Ventas", rot=45, fontsize=12)